# Gavin：未来 15 分钟平均 AC 功率预测、模型评估与误差分析

本 notebook 使用 Lee 的 prediction-ready 特征，但把预测目标改为未来 15 分钟平均 AC 功率：

`mean(total_AC_power_kw from t+1 to t+15)`

这样比预测 `t+15` 单一分钟点值更稳定，也更适合解释短时间运行负载水平。


## 1. 环境与输出路径

使用 pandas/numpy 处理表格，scikit-learn 训练模型。固定随机种子并限制并行线程，便于重复运行。
输出覆盖 `outputs/week7_forecasting_gavin_rf` 中的旧 data/reports/models/figures。


In [1]:
from pathlib import Path
import hashlib
import json
import platform
import shutil
import time
import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display, Markdown
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from threadpoolctl import threadpool_limits

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT = next((p for p in candidates if (p / 'outputs/week7_prediction_ready_dataset').is_dir()), None)
if PROJECT is None:
    PROJECT = Path.cwd() / 'Artificial Intelligence for Smart-Building Log Analysis'
assert (PROJECT / 'outputs/week7_prediction_ready_dataset').is_dir(), '请从项目根目录或 ml 目录运行。'
SOURCE = PROJECT / 'outputs/week7_prediction_ready_dataset'
OUTPUT = PROJECT / 'outputs/week7_forecasting_gavin_rf'
if OUTPUT.exists():
    for folder in ['data', 'reports', 'models', 'figures']:
        shutil.rmtree(OUTPUT / folder, ignore_errors=True)
for folder in ['data', 'reports', 'models']:
    (OUTPUT / folder).mkdir(parents=True, exist_ok=True)
SEED = 42
print('Python:', platform.python_version(), '| sklearn:', sklearn.__version__)
print('输入:', SOURCE)
print('输出:', OUTPUT)


Python: 3.12.7 | sklearn: 1.5.1
输入: d:\IRP\Artificial-Intelligence-for-Smart-Building-Log-Analysis\Artificial Intelligence for Smart-Building Log Analysis\outputs\week7_prediction_ready_dataset
输出: d:\IRP\Artificial-Intelligence-for-Smart-Building-Log-Analysis\Artificial Intelligence for Smart-Building Log Analysis\outputs\week7_forecasting_gavin_rf


## 2. 读取 Lee 的数据和特征清单，并构建 15 分钟平均 target

严格使用 Lee 导出的特征配置。时间戳和 target 不作为模型输入；`zone_category` 使用独热编码，避免把区域编号理解成连续大小关系。

原始 Lee target 是 `t+15` 单点值；本 notebook 重新计算 `t+1` 到 `t+15` 的平均 AC 功率作为预测目标。


In [2]:
config = json.loads((SOURCE / 'reports/preparation_config.json').read_text(encoding='utf-8'))
NUMERIC = config['numeric_features']
CATEGORICAL = config['categorical_features']
FEATURES = NUMERIC + CATEGORICAL
POINT_TARGET = config['target']
TARGET = 'target_total_AC_power_kw_next_15min_mean'
assert config['forecast_horizon_minutes'] == 15

CLEANED_SOURCE = (
    PROJECT / 'outputs/floor6_zone1245_2018_jul01_sep01_total_ac_eda'
    / 'floor6_zone1245_cleaned_features_2018_jul01_sep01.csv.gz'
)
assert CLEANED_SOURCE.is_file(), f'缺少 cleaned source: {CLEANED_SOURCE}'

def future_15min_mean(frame, value_col='total_AC_power_kw'):
    frame = frame.sort_values(['zone', 'timestamp'], kind='stable').copy()
    parts = []
    for _, group in frame.groupby('zone', sort=False):
        future = pd.concat([group[value_col].shift(-i) for i in range(1, 16)], axis=1)
        parts.append(pd.DataFrame({
            'timestamp': group['timestamp'].to_numpy(),
            'zone': group['zone'].to_numpy(),
            TARGET: future.mean(axis=1, skipna=False).to_numpy(),
        }))
    return pd.concat(parts, ignore_index=True)

cleaned_source = pd.read_csv(
    CLEANED_SOURCE,
    usecols=['timestamp', 'zone', 'total_AC_power_kw', 'total_AC_power_kw_was_missing'],
    parse_dates=['timestamp'],
)
cleaned_source.loc[cleaned_source['total_AC_power_kw_was_missing'].astype(bool), 'total_AC_power_kw'] = np.nan
target_lookup = future_15min_mean(cleaned_source)
assert not target_lookup.duplicated(['timestamp', 'zone']).any()

frames = {}
for name in ['train', 'validation', 'test']:
    df = pd.read_csv(
        SOURCE / f'data/prediction_ready_{name}.csv.gz',
        parse_dates=['timestamp', 'target_timestamp'],
        dtype={'zone_category': str},
    )
    df = df.drop(columns=[POINT_TARGET]).merge(target_lookup, on=['timestamp', 'zone'], how='left', validate='one_to_one')
    df = df.dropna(subset=[TARGET]).reset_index(drop=True)
    frames[name] = df

display(pd.DataFrame([{'split': name, 'rows': len(df), 'features': len(FEATURES),
                       'prediction_start': df.timestamp.min(), 'prediction_end': df.timestamp.max(),
                       'target_window_end': df.target_timestamp.max()}
                      for name, df in frames.items()]))
display(frames['train'][['timestamp', 'zone', 'total_AC_power_kw', TARGET]].head())


,split,rows,features,prediction_start,prediction_end,target_end
0,train,253380,29,2018-07-01,2018-08-13 23:44:00,2018-08-13 23:59:00
1,validation,56273,29,2018-08-14,2018-08-23 23:44:00,2018-08-23 23:59:00
2,test,51780,29,2018-08-24,2018-09-01 23:44:00,2018-09-01 23:59:00


,timestamp,zone,total_AC_power_kw,target_total_AC_power_kw_t_plus_15
0,2018-07-01 00:00:00,1,0.0,0.0
1,2018-07-01 00:01:00,1,0.0,0.0
2,2018-07-01 00:02:00,1,0.0,0.0
3,2018-07-01 00:03:00,1,0.0,0.0
4,2018-07-01 00:04:00,1,0.0,0.0


## 2.1 核对 15 分钟平均 target

这里确认新 target 是由 cleaned source 中同一区域未来 15 分钟 AC 功率平均值得到。


In [3]:
target_audit_rows = []
for split, df in frames.items():
    assert df[TARGET].notna().all()
    for zone, group in df.groupby('zone'):
        target_audit_rows.append({
            'split': split,
            'zone': zone,
            'targets_checked': len(group),
            'target_min': group[TARGET].min(),
            'target_max': group[TARGET].max(),
            'target_mean': group[TARGET].mean(),
        })
target_audit = pd.DataFrame(target_audit_rows)
target_audit.to_csv(OUTPUT / 'reports/next15_mean_target_audit.csv', index=False)
display(target_audit)
print('PASS：未来 15 分钟平均 AC 功率 target 已构建完成。')


,zone,AC_channels,channel_count
0,1,z1_AC1(kW),1
1,2,z2_AC1(kW),1
2,4,"z4_AC1(kW), z4_AC2(kW), z4_AC3(kW), z4_AC4(kW)",4
3,5,z5_AC1(kW),1


AssertionError: 目标时间未匹配到原始 AC 总值

## 3. 检查预测对齐与数据泄漏边界


In [ ]:
assert len(FEATURES) == len(set(FEATURES))
assert not set(FEATURES) & {TARGET, POINT_TARGET, 'timestamp', 'target_timestamp', 'baseline_anomaly_label', 'ac_state', 'transition_type'}
assert not any(c.startswith('baseline_anomaly') for c in FEATURES)
for definition in config['splits']:
    name = definition['name']
    df = frames[name]
    assert not df.empty
    assert not df.duplicated(['timestamp', 'floor', 'zone']).any()
    assert df[FEATURES + [TARGET]].notna().all().all()
    assert np.isfinite(df[NUMERIC + [TARGET]].to_numpy(dtype=float)).all()
    assert (df.target_timestamp - df.timestamp).eq(pd.Timedelta(minutes=15)).all()
    assert df.timestamp.ge(pd.Timestamp(definition['prediction_start'])).all()
    assert df.target_timestamp.lt(pd.Timestamp(definition['target_end_exclusive'])).all()
    assert set(df.zone_category).issubset(set(frames['train'].zone_category))
    assert all(g.timestamp.is_monotonic_increasing for _, g in df.groupby('zone'))
assert frames['train'].target_timestamp.max() < frames['validation'].timestamp.min()
assert frames['validation'].target_timestamp.max() < frames['test'].timestamp.min()
print('PASS：输入完整、目标窗口截止为 15 分钟后、时间边界无重叠、输入特征未包含目标。')


## 4. Persistence baseline 与评价指标

**Persistence：ŷ(t+15) = AC(t)**。当前 AC 已在预测时刻可用，所以用它作为未来 15 分钟的预测；这里不使用 AC(t−1)，避免无故多滞后一分钟。

当前 AC 缺失时，沿用 Lee 已导出的因果填补/训练中位数值，因此这是基于 prediction-ready 输入的 persistence。后面另列当前 AC 原始缺失/非缺失样本的误差。

- **MAE**：绝对误差平均值，单位 kW，越小越好。
- **RMSE**：均方误差开根号，单位 kW，对大误差更敏感。
- **Bias**：预测−真实值的平均值，正值代表整体高估。

整体指标按每条区域分钟记录等权计算。AC 接近零时百分比误差不稳定，因此不使用 MAPE。先在验证集建立基线；最终模型确定后再评估测试集。

In [ ]:
def metrics(actual, predicted):
    actual, predicted = np.asarray(actual, dtype=float), np.asarray(predicted, dtype=float)
    assert actual.shape == predicted.shape and actual.ndim == 1 and actual.size > 0
    assert np.isfinite(actual).all() and np.isfinite(predicted).all()
    return {'n': len(actual), 'MAE_kW': mean_absolute_error(actual, predicted),
            'RMSE_kW': np.sqrt(mean_squared_error(actual, predicted)),
            'Bias_kW': float(np.mean(predicted - actual))}

# 用手工可验证的例子检查指标实现。
example = metrics([0, 2], [1, 0])
assert np.isclose(example['MAE_kW'], 1.5)
assert np.isclose(example['RMSE_kW'], np.sqrt(2.5))
assert np.isclose(example['Bias_kW'], -0.5)
validation = frames['validation']
baseline_validation = metrics(validation[TARGET], validation['total_AC_power_kw'])
display(pd.DataFrame([{'split': 'validation', 'model': 'persistence', **baseline_validation}]))

## 5. 训练随机森林候选模型

随机森林由多棵决策树组成，每棵树学习“在这些历史功率、环境和时间条件下，15 分钟后的功率是多少”。最终预测取所有树的平均值，以减少单棵树的波动。

每个候选模型均有 100 棵树。每棵树从训练集有放回抽样 80% 的记录，每次分裂考虑 80% 的特征；比较最大深度 12/20 与叶节点最少样本数 5/20，以控制模型复杂度。随机抽样仅发生在训练集内，不改变按时间划分的验证集和测试集。

只在训练集拟合独热编码器与随机森林；以验证集 MAE 最低选择配置，RMSE 辅助排序。固定随机种子 42，使用 4 个并行线程。模型不需要标准化，也不使用测试集或 9 月 12 日数据调参。

In [ ]:
PARAMETERS = [
    {'n_estimators': 100, 'max_depth': 12, 'min_samples_leaf': 5},
    {'n_estimators': 100, 'max_depth': 20, 'min_samples_leaf': 5},
    {'n_estimators': 100, 'max_depth': 20, 'min_samples_leaf': 20},
]
trained, tuning_rows = {}, []
train = frames['train']
with threadpool_limits(limits=4):
    for i, parameters in enumerate(PARAMETERS, start=1):
        name = f'rf_{i}'
        preprocessor = ColumnTransformer([
            ('numeric', 'passthrough', NUMERIC),
            ('zone', OneHotEncoder(handle_unknown='error', sparse_output=False), CATEGORICAL),
        ])
        model = Pipeline([('features', preprocessor), ('regressor', RandomForestRegressor(
            **parameters, max_features=0.8, bootstrap=True, max_samples=0.8, n_jobs=4, random_state=SEED))])
        started = time.perf_counter()
        model.fit(train[FEATURES], train[TARGET])
        scores = metrics(validation[TARGET], model.predict(validation[FEATURES]))
        tuning_rows.append({'candidate': name, **parameters, **scores,
                            'fit_and_validation_seconds': time.perf_counter() - started})
        trained[name] = model
        print(name, 'validation MAE:', round(scores['MAE_kW'], 4), flush=True)
tuning = pd.DataFrame(tuning_rows).sort_values(['MAE_kW', 'RMSE_kW', 'candidate'])
selected_name = tuning.iloc[0]['candidate']
selected_model = trained[selected_name]
tuning.to_csv(OUTPUT / 'reports/validation_model_selection.csv', index=False)
display(tuning)
print('验证集选定模型:', selected_name)

## 6. 固定模型，生成验证集、测试集与 2018-09-06 预测

每条预测同时保留预测时间 t、窗口截止时间 t+15、未来 15 分钟真实平均值、persistence baseline、RF 预测值和误差。
如果存在 2018-09-06 actual table，也会用同一模型输出该日预测值。


In [ ]:
prediction_inputs = {'validation': frames['validation'], 'test': frames['test']}
SEP6_CANDIDATES = [
    SOURCE / 'data/prediction_ready_2018-09-06_from_raw.csv.gz',
    SOURCE / 'data/prediction_ready_2018-09-06_from_raw.csv',
]
SEP6_FILE = next((path for path in SEP6_CANDIDATES if path.is_file()), None)
if SEP6_FILE is not None:
    sep6 = pd.read_csv(
        SEP6_FILE,
        parse_dates=['timestamp', 'target_timestamp'],
        dtype={'zone_category': str},
    )
    assert TARGET in sep6.columns, f'2018-09-06 actual table 缺少 {TARGET}'
    assert set(FEATURES).issubset(sep6.columns)
    sep6 = sep6.dropna(subset=[TARGET]).reset_index(drop=True)
    prediction_inputs['sep6_2018_09_06'] = sep6
    print('检测到 2018-09-06 actual table:', SEP6_FILE)
else:
    print('未检测到 2018-09-06 actual table，仅输出 validation/test。')

predictions = {}
with threadpool_limits(limits=4):
    for split, df in prediction_inputs.items():
        result = df[['timestamp', 'target_timestamp', 'floor', 'zone', 'total_AC_power_kw_was_missing']].copy()
        result['actual_next15_mean_kW'] = df[TARGET].to_numpy()
        result['persistence_kW'] = df['total_AC_power_kw'].to_numpy()
        result['rf_kW'] = selected_model.predict(df[FEATURES])
        for model_name in ['persistence', 'rf']:
            result[f'{model_name}_error_kW'] = result[f'{model_name}_kW'] - result.actual_next15_mean_kW
        assert len(result) == len(df) and result.notna().all().all()
        predictions[split] = result

joblib.dump(selected_model, OUTPUT / 'models/random_forest.joblib')
for split, result in predictions.items():
    result.to_csv(
        OUTPUT / f'data/{split}_predictions.csv.gz',
        index=False,
        compression={'method': 'gzip', 'mtime': 0},
    )
comparison_preview = predictions['test'].sort_values(['target_timestamp', 'zone']).head(20)
comparison_preview.to_csv(OUTPUT / 'reports/test_prediction_comparison_preview.csv', index=False)
if 'sep6_2018_09_06' in predictions:
    sep6_preview = predictions['sep6_2018_09_06'].sort_values(['target_timestamp', 'zone']).head(20)
    sep6_preview.to_csv(OUTPUT / 'reports/sep6_prediction_comparison_preview.csv', index=False)
display(comparison_preview)
if 'sep6_2018_09_06' in predictions:
    display(sep6_preview)


## 7. 模型效果评估与预测结果对比

分别报告 validation 和 test 的 MAE / RMSE / Bias。相对基线改善率为 `(baseline - ML) / baseline × 100%`，正值代表 ML 更好，负值代表更差；基线误差为零时不定义改善率。

同时保留逐行真实值、persistence 预测、ML 预测和误差，便于直接比较预测趋势与偏差方向。测试集结果才是本次留出评估的最终结果。


In [ ]:
overall = pd.DataFrame([
    {'split': split, 'model': model_name, **metrics(df.actual_next15_mean_kW, df[f'{model_name}_kW'])}
    for split, df in predictions.items() for model_name in ['persistence', 'rf']])
improvements = []
for split in predictions:
    scores = overall[overall.split.eq(split)].set_index('model')
    row = {'split': split}
    for metric in ['MAE_kW', 'RMSE_kW']:
        base = scores.loc['persistence', metric]
        row[f'{metric}_improvement_pct'] = 100 * (base - scores.loc['rf', metric]) / base if base else np.nan
    improvements.append(row)
improvements = pd.DataFrame(improvements)
overall.to_csv(OUTPUT / 'reports/overall_metrics.csv', index=False)
improvements.to_csv(OUTPUT / 'reports/improvement_vs_persistence.csv', index=False)
display(overall.round(4))
display(improvements.round(2))


## 8. 定义用于误差分析的时段

使用**目标时间**划分时段，以说明“哪个时间段的用电最难预测”。这些只是固定日历分组，不代表经过验证的设备启停状态，也不用于重新训练或筛选数据。

工作日：00–07 night_off、07–09 ramp_up、09–17 core_work、17–19 ramp_down、19–24 evening_off；周六/周日全天归为 weekend。边界左闭右开。不推断节假日，沿用原数据时间戳，不进行时区换算。

In [ ]:
PERIOD_ORDER = ['night_off', 'ramp_up', 'core_work', 'ramp_down', 'evening_off', 'weekend']
def operating_period(timestamps):
    hour = timestamps.dt.hour
    return pd.Series(np.select([
        timestamps.dt.dayofweek.ge(5), hour.lt(7), hour.lt(9), hour.lt(17), hour.lt(19)
    ], ['weekend', 'night_off', 'ramp_up', 'core_work', 'ramp_down'], default='evening_off'),
        index=timestamps.index)

boundary_times = pd.Series(pd.to_datetime(['2018-08-27 06:59', '2018-08-27 07:00',
    '2018-08-27 09:00', '2018-08-27 17:00', '2018-08-27 19:00', '2018-08-25 09:00']))
assert operating_period(boundary_times).tolist() == PERIOD_ORDER
for split, df in predictions.items():
    df['target_hour'] = df.target_timestamp.dt.hour
    df['operating_period'] = operating_period(df.target_timestamp)
    df.to_csv(OUTPUT / f'data/{split}_predictions.csv.gz', index=False,
              compression={'method': 'gzip', 'mtime': 0})
test = predictions['test']
display(test.groupby('operating_period').size().reindex(PERIOD_ORDER).rename('test_rows').to_frame())
if 'sep6_2018_09_06' in predictions:
    display(predictions['sep6_2018_09_06'].groupby('operating_period').size().reindex(PERIOD_ORDER).rename('sep6_rows').to_frame())


## 9. 分区域、分时段与缺失场景误差分析

对测试集分别按 zone、目标小时、operating period、zone×period 和当前 AC 原始缺失标记计算指标。每组同时列出 baseline 与 ML，保留样本数 n，避免把样本很少的场景误认为稳定结论。

所有数据均为有效目标记录；缺失或不可用的目标已由 Lee 排除。不同区域功率量级不同，绝对误差较大不一定代表相对表现最差。


In [ ]:
def grouped_metrics(df, columns):
    rows = []
    for keys, group in df.groupby(columns, observed=True, sort=True):
        keys = keys if isinstance(keys, tuple) else (keys,)
        labels = dict(zip(columns, keys))
        for model_name in ['persistence', 'rf']:
            rows.append({**labels, 'model': model_name,
                         **metrics(group.actual_next15_mean_kW, group[f'{model_name}_kW'])})
    result = pd.DataFrame(rows)
    for model_name in ['persistence', 'rf']:
        subset = result[result.model.eq(model_name)]
        assert subset.n.sum() == len(df)
        expected_mae = metrics(df.actual_next15_mean_kW, df[f'{model_name}_kW'])['MAE_kW']
        assert np.isclose(np.average(subset.MAE_kW, weights=subset.n), expected_mae)
    return result

breakdowns = {}
for name, columns in {'zone': ['zone'], 'hour': ['target_hour'], 'period': ['operating_period'],
                       'zone_period': ['zone', 'operating_period'],
                       'current_ac_missing': ['total_AC_power_kw_was_missing']}.items():
    breakdowns[name] = grouped_metrics(test, columns)
    breakdowns[name].to_csv(OUTPUT / f'reports/test_metrics_by_{name}.csv', index=False)
    display(Markdown(f'### Test: {name}'))
    display(breakdowns[name].round(4))


## 10. 自动汇总结论与导出清单

根据真实运行分数生成总结，指出 ML 是否胜过 baseline，以及表现最好/最差的区域和时段。记录源数据 SHA256、软件版本、选参规则和时间分组定义，以便复核。

这些结果只覆盖当前楼层和留出日期，不能直接推断跨楼层或季节泛化。逐分钟数据高度相关，此处不把每分钟当成独立重复实验，也不声称统计显著。时间分组的解释是描述性的，不能据此断定设备故障或误差的因果原因。


In [ ]:
test_scores = overall[overall.split.eq('test')].set_index('model')
gain = improvements[improvements.split.eq('test')].iloc[0]
lines = ['# Gavin forecasting results', '', f'选定模型：{selected_name}（训练集拟合，验证集 MAE 选参）。', '',
         '目标：同一区域未来 15 分钟平均 AC 功率。',
         '本版输出表格、CSV/JSON 报告和模型文件；如存在 2018-09-06 actual table，会额外输出该日 actual vs predicted。', '']
for name in ['persistence', 'rf']:
    row = test_scores.loc[name]
    lines.append(f"- {name}: Test MAE = {row.MAE_kW:.4f} kW; RMSE = {row.RMSE_kW:.4f} kW; Bias = {row.Bias_kW:.4f} kW.")
lines.append(f"- ML 相对 baseline 的 MAE 改善率：{gain['MAE_kW_improvement_pct']:.2f}%；RMSE 改善率：{gain['RMSE_kW_improvement_pct']:.2f}%（负数表示退步）。")
if 'sep6_2018_09_06' in predictions:
    sep6_scores = overall[overall.split.eq('sep6_2018_09_06')].set_index('model')
    for name in ['persistence', 'rf']:
        row = sep6_scores.loc[name]
        lines.append(f"- 2018-09-06 {name}: MAE = {row.MAE_kW:.4f} kW; RMSE = {row.RMSE_kW:.4f} kW; Bias = {row.Bias_kW:.4f} kW.")
for key, column in [('zone', 'zone'), ('period', 'operating_period')]:
    ranking = breakdowns[key].query("model == 'rf'").sort_values('MAE_kW')
    best, worst = ranking.iloc[0], ranking.iloc[-1]
    lines.append(f"- 按 {key} 分组，ML 最低 MAE：{best[column]}（{best.MAE_kW:.4f} kW，n={best.n}）；最高 MAE：{worst[column]}（{worst.MAE_kW:.4f} kW，n={worst.n}）。")
period_compare = breakdowns['period'].pivot(index='operating_period', columns='model', values='MAE_kW')
worse = period_compare.index[period_compare.rf > period_compare.persistence].tolist()
lines.append('ML 的 MAE 高于 persistence 的时段：' + (', '.join(worse) if worse else '无') + '。整体改善不代表所有场景都更好。')
lines += ['', '时段按目标窗口截止时间划分，属于固定日历分组，不代表实测设备启停状态。',
          '总体指标为各区域分钟记录等权；逐行预测对比保存在 data/，摘要对比预览保存在 reports/。',
          '结果仅代表当前楼层与留出时段；缺失目标已由上游排除，不含节假日识别或统计显著性检验。',
          '', '## 产物', '- data/: validation/test 逐行真实值、预测值、误差与时间分组；如存在 sep6 actual table，也包含该日预测。',
          '- reports/: 总体指标、验证选参、预测对比预览、分组指标和运行元数据。',
          '- models/: 包含独热编码器的 RF pipeline。']
lines.append(f'15 分钟平均 target 核对：全部 {target_audit.targets_checked.sum():,} 个 train/validation/test 目标已由 cleaned source 构建。')
summary = '\n'.join(lines)
(OUTPUT / 'reports/results_summary.md').write_text(summary, encoding='utf-8')
display(Markdown(summary))

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

metadata = {'selected_candidate': selected_name, 'selection_metric': 'validation MAE',
            'fit_scope': 'train only; no train+validation refit', 'seed': SEED,
            'features': FEATURES, 'target': TARGET, 'forecast_horizon_minutes': 15,
            'target_definition': 'mean total_AC_power_kw from t+1 to t+15 within the same zone',
            'parameters': selected_model.named_steps['regressor'].get_params(),
            'python': platform.python_version(), 'sklearn': sklearn.__version__,
            'pandas': pd.__version__, 'numpy': np.__version__,
            'operating_period_basis': 'target window end timestamp, weekends first; weekday boundaries 07,09,17,19',
            'baseline': 'prediction-ready total_AC_power_kw at prediction time t',
            'cleaned_source_sha256': sha256(CLEANED_SOURCE),
            'source_sha256': {name: sha256(SOURCE / f'data/prediction_ready_{name}.csv.gz') for name in ['train', 'validation', 'test']}}
if SEP6_FILE is not None:
    metadata['sep6_actual_table_sha256'] = sha256(SEP6_FILE)
(OUTPUT / 'reports/run_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
with threadpool_limits(limits=4):
    restored = joblib.load(OUTPUT / 'models/random_forest.joblib')
    np.testing.assert_allclose(restored.predict(frames['test'][FEATURES].iloc[:100]), test.rf_kW.iloc[:100])
print('PASS：分组样本数与整体 MAE 一致，保存模型重载预测一致。')
display(pd.DataFrame([{'file': str(p.relative_to(OUTPUT)), 'bytes': p.stat().st_size}
                      for p in sorted(OUTPUT.rglob('*')) if p.is_file() and 'figures' not in p.parts]))
